# Arbitrage Time Analysis

This notebook ranks arbitrage opportunities in a historical Everef market-order snapshot using a 45 second per jump travel model.

The table answers:

* what arbitrages exist in the data,
* whether the route can plausibly be travelled within one 15 minute snapshot interval,
* whether the sell side is low, the buy side is high, both, or unclear compared with a volume-weighted market average price,
* expected profit, wallet return, route time, and ISK/hour.

Same-system arbitrages have `gate_jumps = 0`, but are charged one 45 second travel/handling leg so ISK/hour remains finite.

In [ ]:
from arbitrage_time_analysis import AnalysisConfig, build_arbitrage_table, summarize_table

config = AnalysisConfig(
    wallet_amount=100_000_000,
    seconds_per_jump=45,
    snapshot_window_seconds=15 * 60,
    top_n=100,
    per_type_candidate_limit=25,
)

arb_table = build_arbitrage_table(config)
summary = summarize_table(arb_table)

## Summary

`possible_within_15m_snapshot` is `True` when `route_seconds <= 900`. The historical data is only a snapshot, so this is a feasibility heuristic rather than proof that the orders were still available when a pilot arrived.

In [ ]:
summary

## Ranked Arbitrages

Rows are sorted by feasible-first, then descending ISK/hour. `mispricing_type` is based on comparing the sell and buy prices to the item's volume-weighted average price in this snapshot.

In [ ]:
columns = [
    "snapshot_time",
    "item_name",
    "mispricing_type",
    "sell_price",
    "buy_price",
    "market_average_price",
    "from_system",
    "to_system",
    "gate_jumps",
    "route_minutes",
    "possible_within_15m_snapshot",
    "can_take_advantage",
    "feasibility_note",
    "executable_quantity",
    "investment",
    "profit",
    "isk_per_hour",
    "wallet_return_pct",
    "sell_order_id",
    "buy_order_id",
]

arb_table[columns].head(50).style.format({
    "sell_price": "{:,.2f}",
    "buy_price": "{:,.2f}",
    "market_average_price": "{:,.2f}",
    "route_minutes": "{:,.2f}",
    "investment": "{:,.0f}",
    "profit": "{:,.0f}",
    "isk_per_hour": "{:,.0f}",
    "wallet_return_pct": "{:,.2f}",
})

## Save Table

Write the full ranked table to CSV for inspection outside the notebook.

In [ ]:
out = "arbitrage_time_analysis_top100.csv"
arb_table.to_csv(out, index=False)
out